# LeetCode 4030 — Check ASCII Palindromic bằng Python

Link đề: https://leetcode.com/problems/check-ascii-palindromic/ · Độ khó: **Easy** · Chủ đề: String, Bit Manipulation, Two Pointers, Palindrome

Skeleton LeetCode:

```python
class Solution:
    def isPalindromic(self, s: str) -> bool:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đổi **mỗi ký tự** của `s` thành mã ASCII **8 bit** (đệm `0` bên trái), **nối** theo thứ tự, rồi hỏi chuỗi bit dài `8n` có phải **palindrome** không.
2. Biết dùng `ord`, `format(x, '08b')`, `bin`, đảo chuỗi `[::-1]` và kiểm tra palindrome bằng hai con trỏ.
3. Nhận ra **bước đột phá**: chuỗi bit là palindrome khi và chỉ khi `bit_đảo(mã(s[i])) == mã(s[n-1-i])` với mọi `i`; từ đó chỉ có các chữ cái `f`, `n`, `v` (với `n <-> v`) mới có thể xuất hiện.
4. Viết 4 cách (ghép cả chuỗi bit, so từng cặp ký tự bằng đảo 8 bit, bảng ánh xạ `f/n/v`, hai con trỏ) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: không đệm đủ 8 bit, chỉ kiểm tra `s` là palindrome, quên đảo **bit** của từng ký tự, nhầm thứ tự khi đảo cả chuỗi.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ cái thường. Hãy **thay mỗi ký tự** bằng **biểu diễn nhị phân 8 bit của mã ASCII** (đệm số `0` ở bên trái cho đủ 8 bit), rồi **nối** các biểu diễn theo đúng thứ tự ban đầu. Trả về `True` nếu chuỗi bit thu được là **palindrome** (đọc xuôi = đọc ngược), ngược lại `False`.

| Ví dụ | `s` | Output | Giải thích |
|---|---|---|---|
| 1 | `"ff"` | `True` | `f = 102 = 01100110`; chuỗi `0110011001100110` là palindrome |
| 2 | `"leet"` | `False` | `l, e, e, t = 108, 101, 101, 116`, ghép `01101100 01100101 01100101 01110100` không phải palindrome |

Constraints: `1 <= len(s) <= 100`; `s` chỉ gồm chữ cái thường tiếng Anh.

Chuỗi bit dài `8 * len(s) <= 800`.

In [1]:
for s in ("ff", "leet"):
    bits = "".join(format(ord(c), "08b") for c in s)
    print(s, [ord(c) for c in s], bits, "palindrome?", bits == bits[::-1])

ff [102, 102] 0110011001100110 palindrome? True
leet [108, 101, 101, 116] 01101100011001010110010101110100 palindrome? False


## 3. Kiến thức nền

### 3.1. Mã ASCII: `ord` và `chr`

`ord('f') = 102` cho mã của ký tự; `chr(102) = 'f'` đổi ngược. Chữ cái thường `a..z` có mã `97..122`.

In [2]:
print(ord("a"), ord("f"), ord("z"), chr(102))

97 102 122 f


### 3.2. Biểu diễn nhị phân 8 bit: `format(x, '08b')`

`format(102, '08b')` cho `'01100110'`: `b` là nhị phân, `8` là độ rộng tối thiểu, `0` là đệm số `0` bên trái. `bin(102)` cho `'0b1100110'` (có tiền tố `0b` và **không đệm**): chuỗi ngắn hơn, dễ gây sai.

In [3]:
print(format(102, "08b"), bin(102), bin(102)[2:], len(format(102, "08b")))

01100110 0b1100110 1100110 8


### 3.3. Palindrome và đảo chuỗi `[::-1]`

Chuỗi `t` là palindrome nếu `t == t[::-1]`. Chú ý: đảo **cả chuỗi bit** vừa đảo thứ tự các ký tự vừa đảo **bit bên trong từng byte**.

In [4]:
t = "0110011001100110"
print(t == t[::-1], "abca" == "abca"[::-1], "abba" == "abba"[::-1])

True False True


### 3.4. Hai con trỏ kiểm tra palindrome

`l, r = 0, len(t) - 1`; so `t[l]` với `t[r]`, `l += 1`, `r -= 1` đến khi gặp nhau. Dừng sớm ở lần khác nhau đầu tiên: nhanh hơn dựng chuỗi đảo (nhưng cùng `O(n)`).

In [5]:
def is_pal(t):
    l, r = 0, len(t) - 1
    while l < r:
        if t[l] != t[r]:
            return False
        l += 1
        r -= 1
    return True

print(is_pal("0110"), is_pal("0111"), is_pal("1"))

True False True


### 3.5. Đảo bit của một byte: `int(format(x, '08b')[::-1], 2)`

Đảo chuỗi 8 bit rồi đổi lại số: `rev(102) = 102` (vì `01100110` đối xứng), `rev(110)`: `110 = 01101110 -> 01110110 = 118`. Hàm này là chìa khóa để so sánh **từng cặp ký tự đối xứng** mà không cần dựng cả chuỗi bit.

In [6]:
def rev8(x):
    return int(format(x, "08b")[::-1], 2)

print(rev8(102), rev8(ord("n")), chr(rev8(ord("n"))), rev8(ord("a")))

102 118 v 134


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cách trực tiếp: dựng chuỗi bit `B` của `s` rồi so `B == B[::-1]`.

Nhưng có cách nhìn thông minh hơn. Viết `B = b(s_0) b(s_1) ... b(s_{n-1})` (`b(c)` là 8 bit của `c`). Đảo cả chuỗi:

`reverse(B) = rev(b(s_{n-1})) rev(b(s_{n-2})) ... rev(b(s_0))`

(đảo thứ tự các byte **và** đảo bit trong từng byte). `B` là palindrome khi và chỉ khi hai chuỗi này **bằng nhau**, tức với mọi `i`:

`b(s_i) = rev(b(s_{n-1-i}))`.

Tính tay `"ff"`: `b(f) = 01100110`, `rev(b(f)) = 01100110` (đối xứng) -> `True`.
Tính tay `"nv"`: `b(n) = 01101110`, `b(v) = 01110110`; `rev(b(v)) = 01101110 = b(n)` ✓ nên `True`. (Đây là cặp "bù nhau".)
Tính tay `"ab"`: `b(a) = 01100001`, `rev(b(b)) = rev(01100010) = 01000110 ≠ b(a)` -> `False`.

| `s` | các byte | điều kiện | kết quả |
|---|---|---|---|
| `"ff"` | `01100110`, `01100110` | `b(f) = rev(b(f))` | `True` |
| `"nv"` | `01101110`, `01110110` | `b(n) = rev(b(v))` | `True` |
| `"ab"` | `01100001`, `01100010` | `b(a) != rev(b(b))` | `False` |

In [7]:
def bits(s):
    return "".join(format(ord(c), "08b") for c in s)

for s in ("ff", "nv", "ab", "vn", "nn"):
    B = bits(s)
    print(s, B, B == B[::-1])

ff 0110011001100110 True
nv 0110111001110110 True
ab 0110000101100010 False
vn 0111011001101110 True
nn 0110111001101110 False


## 5. Cách 1 — Brute Force / baseline: dựng chuỗi bit rồi so với bản đảo

1. `B = "".join(format(ord(c), "08b") for c in s)`.
2. Trả `B == B[::-1]`.

Đúng theo đúng định nghĩa, dài `8n` ký tự. "Nút thắt": ta xử lý `8n` bit trong khi chỉ cần so `n` cặp ký tự (nhìn mục 6).

In [8]:
class SolutionBits:
    def isPalindromic(self, s):
        B = "".join(format(ord(c), "08b") for c in s)
        return B == B[::-1]

print(SolutionBits().isPalindromic("ff"), SolutionBits().isPalindromic("leet"))

True False


**Độ phức tạp:** Time `O(n)` (`8n` bit), Space `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá (so cặp ký tự):** `B` palindrome <=> với mọi `i`: `ord(s[i]) == rev8(ord(s[n-1-i]))`.

**Bước đột phá thứ hai (chỉ còn 3 chữ cái):** để `rev8(c)` là một chữ cái thường (mã `97..122`, dạng `011xxxxx`), ta cần bit 0 của `c` bằng `0` (để bit 7 của `rev8` là `0`), bit 1 bằng `1`, bit 2 bằng `1`: tức `c` có ba bit thấp `110`, `c % 8 == 6`. Trong `97..122` chỉ có `102 = 'f'`, `110 = 'n'`, `118 = 'v'`. Tính `rev8`:

- `rev8('f') = 'f'` (`01100110` đối xứng),
- `rev8('n') = 'v'` (`01101110 -> 01110110`),
- `rev8('v') = 'n'`.

**Hệ quả:** `s` chỉ được chứa `f`, `n`, `v`; và `s[i]` phải là **ảnh qua ánh xạ `f->f, n->v, v->n`** của `s[n-1-i]`. Ví dụ `"fnvf"`? `s[0]='f'` ↔ `s[3]='f'` ✓; `s[1]='n'` ↔ `s[2]='v'` ✓ -> `True`.

**Invariant:** hai con trỏ `l`, `r` từ hai đầu; mọi cặp đã xét thỏa `s[l] = M(s[r])`, với `M` là ánh xạ trên (và `M(c)` không xác định nếu `c` ngoài `{f, n, v}` -> `False`).

## 7. Các cách tiếp theo

### Cách 2 — So từng cặp ký tự đối xứng bằng `rev8`

Không dựng chuỗi bit dài: với mọi `i` kiểm tra `ord(s[i]) == rev8(ord(s[n-1-i]))`. Nhanh gấp tám lần về số phép so sánh.

In [9]:
def rev8(x):
    return int(format(x, "08b")[::-1], 2)

class SolutionPairs:
    def isPalindromic(self, s):
        n = len(s)
        return all(ord(s[i]) == rev8(ord(s[n - 1 - i])) for i in range(n))

print(SolutionPairs().isPalindromic("ff"), SolutionPairs().isPalindromic("leet"))

True False


**Độ phức tạp:** Time `O(n)` (mỗi cặp `O(8)`), Space `O(1)`.

### Cách 3 — Bảng ánh xạ chỉ gồm `f`, `n`, `v`

Dựng sẵn `MIRROR = {'f': 'f', 'n': 'v', 'v': 'n'}` (tính bằng `rev8` cho cả 26 chữ cái rồi giữ các chữ ánh xạ ra chữ cái). Kiểm tra `s[i]` có trong `MIRROR` và `MIRROR[s[i]] == s[n-1-i]`.

In [10]:
MIRROR = {chr(c): chr(rev8(c)) for c in range(97, 123) if 97 <= rev8(c) <= 122}
print(MIRROR)

class SolutionMirror:
    def isPalindromic(self, s):
        n = len(s)
        for i in range(n):
            if s[i] not in MIRROR or MIRROR[s[i]] != s[n - 1 - i]:
                return False
        return True

print(SolutionMirror().isPalindromic("ff"), SolutionMirror().isPalindromic("leet"))

{'f': 'f', 'n': 'v', 'v': 'n'}
True False


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (bảng 3 phần tử, tính một lần).

### Cách 4 — Hai con trỏ trên chuỗi bit, dừng sớm

Vẫn dựng `B` nhưng dùng hai con trỏ (dừng ở lần khác nhau đầu tiên). Không nhanh hơn về bậc, nhưng thoát sớm.

In [11]:
class SolutionTwoPointers:
    def isPalindromic(self, s):
        B = "".join(format(ord(c), "08b") for c in s)
        l, r = 0, len(B) - 1
        while l < r:
            if B[l] != B[r]:
                return False
            l += 1
            r -= 1
        return True

print(SolutionTwoPointers().isPalindromic("ff"), SolutionTwoPointers().isPalindromic("leet"))

True False


**Độ phức tạp:** Time `O(n)` (best `O(1)`), Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 với `s = "nv"` (`n = 2`):

| `i` | `s[i]` | `ord` | `s[n-1-i]` | `rev8(ord(s[n-1-i]))` | bằng? |
|---|---|---|---|---|---|
| 0 | `n` | 110 | `v` | `rev8(118) = 110` | có |
| 1 | `v` | 118 | `n` | `rev8(110) = 118` | có |

Hai cặp đều khớp -> `True`. Chuỗi bit đầy đủ `0110111001110110` cũng là palindrome.

In [12]:
def trace_pairs(s):
    n = len(s)
    print("s =", s)
    for i in range(n):
        a = ord(s[i])
        b = rev8(ord(s[n - 1 - i]))
        print(f"  i={i}: ord({s[i]!r})={a} ({a:08b}); rev8(ord({s[n - 1 - i]!r}))={b} ({b:08b}) -> {'khớp' if a == b else 'KHÔNG khớp'}")
        if a != b:
            return False
    return True

assert trace_pairs("nv") is True
assert trace_pairs("ab") is False
assert trace_pairs("ff") is True

s = nv
  i=0: ord('n')=110 (01101110); rev8(ord('v'))=110 (01101110) -> khớp
  i=1: ord('v')=118 (01110110); rev8(ord('n'))=118 (01110110) -> khớp
s = ab
  i=0: ord('a')=97 (01100001); rev8(ord('b'))=70 (01000110) -> KHÔNG khớp
s = ff
  i=0: ord('f')=102 (01100110); rev8(ord('f'))=102 (01100110) -> khớp
  i=1: ord('f')=102 (01100110); rev8(ord('f'))=102 (01100110) -> khớp


In [13]:
def trace_bits(s):
    B = bits(s)
    print(f"s={s}: B = {B}")
    print(f"     B[::-1] = {B[::-1]}  -> palindrome? {B == B[::-1]}")
    return B == B[::-1]

assert trace_bits("nv") is True and trace_bits("nn") is False

s=nv: B = 0110111001110110
     B[::-1] = 0110111001110110  -> palindrome? True
s=nn: B = 0110111001101110
     B[::-1] = 0111011001110110  -> palindrome? False


## 9. Code nộp LeetCode

Dùng Cách 2 (so cặp ký tự bằng đảo 8 bit): ngắn, nhanh và không dựng chuỗi dài.

In [14]:
from typing import List, Optional


class Solution:
    def isPalindromic(self, s: str) -> bool:
        def rev8(x: int) -> int:
            return int(format(x, "08b")[::-1], 2)

        n = len(s)
        return all(ord(s[i]) == rev8(ord(s[n - 1 - i])) for i in range(n))


print(Solution().isPalindromic("ff"), Solution().isPalindromic("leet"))

True False


## 10. Vì sao thuật toán đúng

Gọi `b(c)` là chuỗi 8 bit của `ord(c)` và `rev(w)` là chuỗi `w` viết ngược. `B = b(s_0) b(s_1) ... b(s_{n-1})`.

1. **Đảo cả chuỗi.** `rev(B) = rev(b(s_{n-1})) rev(b(s_{n-2})) ... rev(b(s_0))` vì đảo một chuỗi nối bằng cách đảo từng khối rồi đảo thứ tự khối.
2. **So sánh theo khối.** Cả `B` và `rev(B)` gồm `n` khối cùng dài 8 bit. Chúng bằng nhau khi và chỉ khi mọi khối tương ứng bằng nhau: khối `i` của `B` là `b(s_i)`, khối `i` của `rev(B)` là `rev(b(s_{n-1-i}))`. Vậy `B` palindrome <=> `b(s_i) = rev(b(s_{n-1-i}))` với mọi `i`.
3. **Chuyển về số.** `b(s_i)` là mã 8 bit của `ord(s_i)` và `rev(b(c))` là mã 8 bit của số `rev8(ord(c))`; hai chuỗi 8 bit bằng nhau iff hai số bằng nhau. Do đó điều kiện là `ord(s_i) == rev8(ord(s_{n-1-i}))` (Cách 2).
4. **Tính hữu hạn của ánh xạ (Cách 3).** `ord(s_i) in [97, 122]` nên `rev8(ord(s_{n-1-i}))` phải thuộc `[97, 122]`: chỉ `f, n, v` thỏa (mục 6). Với chúng, `rev8` cho `f->f, n->v, v->n`.
5. **Điều kiện cho ký tự giữa.** Với `n` lẻ, `i = n-1-i` ở giữa: cần `ord(s_i) = rev8(ord(s_i))`, tức byte tự đối xứng; trong `97..122` chỉ `f`. Thuật toán tự xử lý vì cặp `(i, i)` cũng được kiểm tra (và với `'n'` hay `'v'` ở giữa thì `rev8(n) = v != n` nên `False`).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) <= 100`.

- Cách 1: Time `O(n)` (xử lý `8n` bit), Space `O(n)`.
- Cách 2 (nộp): Time `O(n)` (mỗi cặp `O(1)` vì đảo 8 bit), Space `O(1)`.
- Cách 3: Time `O(n)`, Space `O(1)` (bảng ba phần tử).
- Cách 4: Time `O(n)`, Space `O(n)`.

Không thể dưới `O(n)` vì phải xem mọi ký tự (kết quả `True` cần xác nhận cả `n` ký tự).

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionBits(), SolutionPairs(), SolutionMirror(), SolutionTwoPointers(), Solution()]

fixed = [
    ("ff", True), ("leet", False), ("f", True), ("n", False), ("v", False), ("nv", True), ("vn", True),
    ("nn", False), ("vv", False), ("a", False), ("fnvf", True), ("fvnf", True), ("fnnf", False),
    ("fff", True), ("nfv", True), ("vfn", True), ("nfn", False), ("fa", False), ("abc", False),
    ("f" * 100, True), ("nv" * 50, True), ("n" * 50 + "v" * 50, True),
]
for s, expected in fixed:
    for sol in solutions:
        assert sol.isPalindromic(s) == expected, (type(sol).__name__, s)
print("test cố định OK:", len(fixed), "ca")

random.seed(4030)
true_count = 0
for _ in range(1500):
    n = random.randint(1, 100)
    if random.random() < 0.5:                       # sinh palindrome thật sự từ nửa trái
        half = [random.choice("fnv") for _ in range(n // 2)]
        mid = ["f"] if n % 2 else []
        M2 = {"f": "f", "n": "v", "v": "n"}
        s = "".join(half + mid + [M2[c] for c in reversed(half)])
    else:
        alphabet = random.choice(["fnv", "fnva", "abcdefghijklmnopqrstuvwxyz"])
        s = "".join(random.choice(alphabet) for _ in range(n))
    expected = bits(s) == bits(s)[::-1]             # trọng tài: theo đúng định nghĩa
    true_count += expected
    for sol in solutions:
        assert sol.isPalindromic(s) == expected, (type(sol).__name__, s)
print("test ngẫu nhiên OK, số ca True:", true_count)
assert true_count > 200

# vét cạn mọi chuỗi độ dài <= 3 trên bảng chữ cái đủ 26 chữ
from itertools import product
for L in (1, 2, 3):
    for tup in product("abcfnvxz", repeat=L):
        s = "".join(tup)
        assert Solution().isPalindromic(s) == (bits(s) == bits(s)[::-1]), s
print("vét cạn độ dài <= 3 OK")

test cố định OK: 22 ca
test ngẫu nhiên OK, số ca True: 783
vét cạn độ dài <= 3 OK


### Test thuộc tính (property test)

(1) Nếu `s` là palindrome bit thì `s` chỉ gồm `f, n, v`. (2) Đảo ánh xạ: `reverse` rồi đổi `n <-> v` cho cùng kết quả `True`. (3) Nối hai palindrome bit (`a + rev_map(a)`) vẫn là palindrome bit. (4) Thêm `f` vào hai đầu giữ nguyên kết quả.

In [16]:
random.seed(40300)
SWAP = str.maketrans("nv", "vn")
for _ in range(600):
    n = random.randint(1, 30)
    s = "".join(random.choice("fnv") for _ in range(n))
    r = Solution().isPalindromic(s)
    if r:
        assert set(s) <= set("fnv")
        assert s == s[::-1].translate(SWAP)
    assert r == (s == s[::-1].translate(SWAP))
    assert Solution().isPalindromic("f" + s + "f") == r
    assert Solution().isPalindromic(s + s[::-1].translate(SWAP)) is True
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: kiểm tra `s` là palindrome (không phải chuỗi bit)

`"nn"` là palindrome chữ cái nhưng chuỗi bit `0110111001101110` không phải.

In [17]:
def wrong_string_palindrome(s):
    return s == s[::-1]                                          # SAI: kiểm tra chuỗi ký tự

print(wrong_string_palindrome("nn"), "!=", Solution().isPalindromic("nn"))
assert wrong_string_palindrome("nn") is True and Solution().isPalindromic("nn") is False
assert wrong_string_palindrome("nv") is False and Solution().isPalindromic("nv") is True

True != False


### Lỗi 2: không đệm đủ 8 bit (dùng `bin(...)[2:]`)

Chữ cái thường có 7 chữ số nhị phân khi bỏ số `0` đứng đầu (`97 = 1100001`). Bỏ đệm thì ranh giới giữa các ký tự lệch và đáp án đổi: `"nv"` thực sự là `True` nhưng chuỗi 7 bit cho `False`.

In [18]:
def wrong_no_pad(s):
    B = "".join(bin(ord(c))[2:] for c in s)                      # SAI: 7 bit, không đệm
    return B == B[::-1]

print(wrong_no_pad("nv"), "!=", Solution().isPalindromic("nv"))
assert wrong_no_pad("nv") is False and Solution().isPalindromic("nv") is True

False != True


### Lỗi 3: chỉ đảo thứ tự ký tự, quên đảo **bit** bên trong từng byte

So `ord(s[i]) == ord(s[n-1-i])` chỉ khớp khi `s` là palindrome ký tự. Với `"nv"` ký tự khác nhau nhưng chuỗi bit palindrome.

In [19]:
def wrong_no_bit_reverse(s):
    n = len(s)
    return all(ord(s[i]) == ord(s[n - 1 - i]) for i in range(n))  # SAI: quên rev8

print(wrong_no_bit_reverse("nv"), "!=", Solution().isPalindromic("nv"))
assert wrong_no_bit_reverse("nv") is False and Solution().isPalindromic("nv") is True
assert wrong_no_bit_reverse("nn") is True and Solution().isPalindromic("nn") is False

False != True


### Lỗi 4: đảo bit của mọi ký tự nhưng quên đảo thứ tự ký tự

Chỉ so `ord(s[i]) == rev8(ord(s[i]))` (mỗi ký tự tự đối xứng) bỏ qua mối liên hệ giữa `s[i]` và `s[n-1-i]`.

In [20]:
def wrong_no_order_reverse(s):
    return all(ord(c) == rev8(ord(c)) for c in s)                 # SAI: chỉ ký tự tự đối xứng ('f')

print(wrong_no_order_reverse("fa"), wrong_no_order_reverse("ff"), wrong_no_order_reverse("nv"))
assert wrong_no_order_reverse("ff") is True and wrong_no_order_reverse("nv") is False
assert Solution().isPalindromic("nv") is True

False True False


### Lỗi 5: dùng độ rộng `'07b'` hoặc `'09b'` (sai số bit)

Đề nói **8 bit**. Độ rộng khác đổi chuỗi bit và có thể đổi kết quả (nhất là kiểm tra bằng hai con trỏ).

In [21]:
def wrong_width(s, w=9):
    B = "".join(format(ord(c), f"0{w}b") for c in s)              # SAI: 9 bit
    return B == B[::-1]

print(wrong_width("ff"), wrong_width("nv"), "| đúng:", Solution().isPalindromic("ff"), Solution().isPalindromic("nv"))
assert wrong_width("nv") is False and Solution().isPalindromic("nv") is True

False False | đúng: True True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Palindrome của chuỗi sau biến đổi" -> hãy tìm **dạng tương đương trên chuỗi gốc**: `B` palindrome <=> mỗi cặp đối xứng thỏa `f(s[i]) = s[n-1-i]` (ở đây `f` là đảo 8 bit).
- Đảo một chuỗi nối = đảo thứ tự các khối **và** đảo từng khối.
- `format(x, '08b')` là cách an toàn để có 8 bit; `bin` không đệm.
- Trước khi viết thuật toán phức tạp, **khảo sát miền nhỏ** (26 chữ cái): ở đây chỉ `f, n, v` có thể có mặt -> kiểm tra tức thì.
- Kiểm tra bằng trọng tài đơn giản (dựng chuỗi bit đầy đủ) trên nhiều chuỗi ngẫu nhiên.
- Ca biên: một ký tự (`"f"` True, `"n"` False), chuỗi toàn `f`, cặp `nv`.

## 15. Biến thể của bài

1. **Valid Palindrome** (125), **Palindrome Number** (9), **Palindromic Substrings** (647): palindrome cơ bản.
2. **Reverse Bits** (190): đảo bit của số nguyên 32 bit (cùng kỹ thuật `rev`).
3. **Binary Number with Alternating Bits** (693): bit liền kề.
4. Biến thể: dùng **mã Unicode 16 bit** hoặc 7 bit ASCII; viết lại điều kiện với `rev16` / `rev7`.
5. Biến thể: chuỗi có cả chữ hoa/số: bảng ánh xạ có nhiều ký tự hơn (tính bằng chương trình).
6. Biến thể: đếm số xâu độ dài `L` trên `a..z` mà chuỗi bit là palindrome: kết quả `3^{L/2}` (chẵn) hoặc `3^{(L-1)/2}` (lẻ, vì chữ giữa chỉ `f`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chuỗi bit đầy đủ | `B == B[::-1]` | `O(n)` | `O(n)` | Theo đúng định nghĩa, trọng tài |
| 2. So cặp bằng `rev8` | `ord(s[i]) == rev8(ord(s[n-1-i]))` | `O(n)` | `O(1)` | **Nộp bài** |
| 3. Bảng `f/n/v` | ánh xạ `f->f, n->v, v->n` | `O(n)` | `O(1)` | Khi muốn kết quả "đọc được", ngắn |
| 4. Hai con trỏ trên bit | thoát sớm khi khác | `O(n)` | `O(n)` | Dữ liệu dài, hay sai sớm |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().isPalindromic("fvnf")` và `Solution().isPalindromic("fnf")` ra gì?

**Bài 2 (tính tay):** viết 8 bit của `'n'` và `'v'`, đảo bit rồi giải thích vì sao chúng "bù nhau".

**Bài 3 (điền TODO):** hoàn thành `all_palindromic_letters()` trả về danh sách mọi chữ cái `c` mà `"c"` (một ký tự) có chuỗi bit palindrome.

**Bài 4 (tự viết lại):** viết `count_palindromic(L)`: số chuỗi độ dài `L` trên `a..z` có chuỗi bit palindrome (dùng công thức rồi kiểm với `L <= 3` bằng vét cạn).

**Bài 5 (mini challenge):** viết `first_mismatch(s)`: vị trí cặp `(i, n-1-i)` đầu tiên không khớp, hoặc `-1` nếu là palindrome bit.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().isPalindromic("fvnf"), Solution().isPalindromic("fnf"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def all_palindromic_letters_todo():
    # TODO: duyệt 26 chữ cái, giữ chữ c nếu bits(c) == bits(c)[::-1]
    return []

True False


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: dùng bảng `f->f, n->v, v->n`. Bài 2: `n = 01101110`, `v = 01110110`.

**Mức 2 (vừa):** Bài 3: chỉ cần chuỗi bit 8 ký tự tự đối xứng. Bài 4: nửa trái chọn tự do trong `{f, n, v}`: `3^(L//2)` cách; nếu `L` lẻ chữ giữa phải là `f`. Bài 5: vòng `for i in range(n)`.

**Mức 3 (gần đáp án):** Bài 1: `"fvnf"` là `True` (cặp ngoài `f..f`, cặp trong `v..n` vì `rev8(n) = v`); `"fnf"` là `False` (chữ giữa `n` không tự đối xứng). Bài 3: `['f']`. Bài 4: `count_palindromic(1) = 1`, `(2) = 3`, `(3) = 3`. Bài 5: `first_mismatch("ab") = 0`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().isPalindromic("fvnf") is True and Solution().isPalindromic("fnf") is False
assert format(ord("n"), "08b") == "01101110" and format(ord("v"), "08b") == "01110110"
assert format(ord("n"), "08b")[::-1] == format(ord("v"), "08b")

# Bài 3
def all_palindromic_letters():
    return [chr(c) for c in range(97, 123) if bits(chr(c)) == bits(chr(c))[::-1]]

assert all_palindromic_letters() == ["f"]

# Bài 4
def count_palindromic(L):
    return 3 ** (L // 2)          # nửa trái chọn tự do trong {f, n, v}; chữ giữa (nếu có) buộc là 'f'

assert [count_palindromic(L) for L in (1, 2, 3)] == [1, 3, 3]
from itertools import product
for L in (1, 2, 3):
    brute = sum(1 for tup in product("abcdefghijklmnopqrstuvwxyz", repeat=L) if Solution().isPalindromic("".join(tup)))
    assert brute == count_palindromic(L), (L, brute)

# Bài 5
def first_mismatch(s):
    n = len(s)
    for i in range(n):
        if ord(s[i]) != rev8(ord(s[n - 1 - i])):
            return i
    return -1

assert first_mismatch("ab") == 0 and first_mismatch("ff") == -1 and first_mismatch("fnvx") == 0
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def isPalindromic(self, s: str) -> bool:
        def rev8(x: int) -> int:
            return int(format(x, "08b")[::-1], 2)      # đảo 8 bit của một byte

        n = len(s)
        # chuỗi bit palindrome <=> mỗi cặp đối xứng: byte(s[i]) == đảo-bit(byte(s[n-1-i]))
        return all(ord(s[i]) == rev8(ord(s[n - 1 - i])) for i in range(n))
```

Khung chung "palindrome sau biến đổi": đưa điều kiện về **từng cặp đối xứng** của chuỗi gốc.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Dùng `format(x, "08b")` (đủ 8 bit, có đệm `0`), đảo **cả thứ tự ký tự lẫn bit trong từng byte**.
- [ ] Đã thử `"ff"` (True), `"nv"` (True), `"nn"` (False), một ký tự `"f"` / `"n"`, và so với chuỗi bit đầy đủ.